# exp073_gpu_reproducibility_guard_for_exp063_full_replay inference

Saved-booster inference for the exp073 full replay end-to-end reproducibility guard. This notebook regenerates exp063 public replay PF/Beam/likelihood-PF test features from the current raw test files with stable per-well seeds, then applies the exp073 saved boosters.


## Contents

1. Setup and configuration
2. Source and raw test check
3. Test feature replay and saved booster inference
4. Metrics and submission


## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from exp063_full_replay_reproducibility_guard import (
    TRACKER_TEST_FEATURES,
    find_model_manifest,
    run_saved_model_inference,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Inference mode:", cfg_get(config, "inference.mode"))
print("Deterministic PF:", cfg_get(config, "inference.deterministic_pf"))
print("Seed policy:", cfg_get(config, "inference.seed_policy"))
print("Selected mode:", cfg_get(config, "inference.selected_mode"))
print("Selected model:", cfg_get(config, "inference.selected_model"))
print("Inference kernel sources:", cfg_get(config, "runtime.kaggle.inference_kernel_sources"))


## 2. Source and raw test check


In [ ]:
manifest_path = find_model_manifest(cfg_get(config, "inference.model_manifest_path"))
print("Model manifest:", manifest_path)
print("Raw data dir:", paths.raw_data_dir)
print("Raw test dir:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Regenerate test features:", cfg_get(config, "inference.regenerate_test_features", True))
print("Feature generation n_jobs:", cfg_get(config, "inference.feature_generation.n_jobs"))
print("Expected generated full replay test feature file:", paths.artifacts_dir / TRACKER_TEST_FEATURES)

test_files = sorted(paths.test_data_dir.glob("*__horizontal_well.csv"))
print("Test wells:", len(test_files))
print("First test files:", [path.name for path in test_files[:5]])
display(pd.read_csv(paths.sample_submission_path, nrows=5, dtype={"id": str}))


## 3. Test feature replay and saved booster inference


In [ ]:
summary = run_saved_model_inference(
    output_dir=paths.artifacts_dir,
    submission_path=paths.submission_path,
    sample_submission_path=paths.sample_submission_path,
    data_dir=paths.raw_data_dir,
    tracker_test_path=cfg_get(config, "data.exp063_tracker_features_test_local"),
    model_manifest_path=cfg_get(config, "inference.model_manifest_path"),
    mode_name=str(cfg_get(config, "inference.selected_mode", "gpu_repro_guard_dp_threads8")),
    model_name=str(cfg_get(config, "inference.selected_model", "lgb_mean")),
    submission_target_column=str(cfg_get(config, "data.submission_target_column", "tvt")),
    regenerate_test_features=bool(cfg_get(config, "inference.regenerate_test_features", True)),
    n_jobs=int(cfg_get(config, "inference.feature_generation.n_jobs", 8)),
    pf_seeds=int(cfg_get(config, "inference.feature_generation.pf_seeds", 128)),
    pf_particles=int(cfg_get(config, "inference.feature_generation.pf_particles", 500)),
    fast=bool(cfg_get(config, "inference.feature_generation.fast", False)),
    use_gpu=str(cfg_get(config, "inference.feature_generation.use_gpu", "auto")),
)
print(json.dumps(summary, indent=2))


## 4. Metrics and submission


In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "exp063_full_replay_repro_guard_inference_metrics.csv")
submission = pd.read_csv(paths.submission_path)

display(metrics)
display(submission.head())
print("Submission:", paths.submission_path, "rows=", len(submission))
